### 03 - Scanned PDF NER
Pipeline: PDF -> PyMuPDF -> GLiNER -> predicted entities -> strict/relaxed evaluation.

In [1]:
import sys
sys.path.append('..')

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.inference import load_gliner_model, predict_dataset
from ner.pdf import extract_pdf_dataset
from ner.utils import load_jsonl, save_responses_to_jsonl


C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
PDF_DIR = '../data/pdf_scanned'
OUTPUT_PATH = '../results/pdf_scanned/gliner_small_validation.jsonl'

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]


In [3]:
gold_samples = load_jsonl(VALIDATION_PATH)[:10]   # solo tenemos 10
pdf_samples = extract_pdf_dataset(gold_samples, PDF_DIR, suffix="_scanned")

# Comprobamos que se ha extraído texto de los PDF
print(len(pdf_samples))
print(pdf_samples[0]["text"][:500])


10



In [4]:
model = load_gliner_model()
responses = predict_dataset(model, pdf_samples, LABELS, threshold=0.5)
save_responses_to_jsonl(responses, OUTPUT_PATH)

C:\Users\garci\Desktop\Projecte_Name_Entity_Recognition-master\Projecte_Name_Entity_Recognition-master\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [5]:
strict_results = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed_results = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)



STRICT EVALUATION
Micro -> P: 0.00 | R: 0.00 | F1: 0.00 | TP: 0 | FP: 0 | FN: 410

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 0.00   0.00   0.00       0       0     100
ID_CARD_NUMBER              0.00   0.00   0.00       0       0      40
ADDRESS                     0.00   0.00   0.00       0       0      40
REGISTRATION_NUMBER         0.00   0.00   0.00       0       0      50
DATE                        0.00   0.00   0.00       0       0      80
AMOUNT                      0.00   0.00   0.00       0       0      40
ORGANIZATION                0.00   0.00   0.00       0       0      60

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 0.00 | R: 0.00 | F1: 0.00 | TP: 0 | FP: 0 | FN: 410

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------------------------------------------------------------------------

In [ ]:
##Todas las métricas salen a 0 (TP = 0, FP = 0, FN = 410). Esto no se debe a un fallo de GLiNER ni de la evaluación, sino a que **PyMuPDF no extrae texto de estos PDF** comparado con la task 2 que hay texto plano en el pdf. Apartir de este problema le pregunte al chatgpt como conseguir datos y me dijo lo siguiente (Para obtener texto de estos documentos sería necesario aplicar OCR)